In [0]:
%pip install google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 74.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import json
import google.generativeai as genai

# 1. SETUP & LOAD DATA
genai.configure(api_key="INSERT-API-KEY")

# Reading from a Unity Catalog Volume (file path)
input_path = "/Volumes/student_system/performance/student_grade/grades.csv"
df_spark = spark.read.csv(input_path, header=True, inferSchema=True)

csv_data_context = df_spark.toPandas().to_csv(index=False)

# 2. DEFINE THE AGENT

prompt = f"""
Act as an expert Data Analyst. Analyze this student performance data, that was provided in CSV format

OBJECTIVE:
Identify 'At Risk' students based on the following logic.
If a student meet multiple risks, prioritize the most severe systemic issue first (Attendance > Average Score > Single Subject).

RISK CRITERIA (Apply in this order):
1. Attendance Risk: 
   - Rule: Attendance < 50%
   - Intervention: "Attendance plan to strengthen student engagement, accompanied by parents and tutors."

2. Average Academic Risk (Low Average):
   - Rule: avg_score < 50
   - Intervention: "Comprehensive academic support to identify opportunities for improvement."

3. Specific Subject Risk:
   - Rule: Score < 50 in any individual subject
   - Intervention: "Targeted Tutoring."

OUTPUT FORMAT:
Return a STRICT JSON list of objects. Do not include markdown formatting.
Include only students who trigger a risk.
Keys:
- "Student_Name"
- "Risk_Reason" (e.g., "Low Average Score: 42" or "Low Math Score: 45" or "Attendance Risk: 45%")
- "Recommended_Intervention"

DATA:
{csv_data_context}
"""

# 3. RUN WITH JSON MODE
model = genai.GenerativeModel("gemini-2.5-flash")

# --- OPTIMIZATION 2: OUTPUT FORMAT (JSON Mode) ---
# We force the model to return a clean JSON string.
# This eliminates the need for .replace("```json", "")
response = model.generate_content(
    prompt,
    generation_config={"response_mime_type": "application/json"}
)

# 4. PARSING & SAVE
# Since we used JSON mode, we can read the text directly.
pdf_result = pd.read_json(response.text)

# Save back to the Catalog
final_spark_df = spark.createDataFrame(pdf_result)
target_table = "student_system.performance.at_risk_students"
final_spark_df.write.mode("overwrite").saveAsTable(target_table)

print(f"Success! Saved to {target_table}")
display(final_spark_df)

/home/spark-ae073dc4-7d2c-417b-995d-e0/.ipykernel/2536/command-7504378486571600-1725454830:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
/home/spark-ae073dc4-7d2c-417b-995d-e0/.ipykernel/2536/command-7504378486571600-1725454830:61: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pdf_result = pd.read_json(response.text)


Success! Saved to student_system.performance.at_risk_students


Student_Name,Risk_Reason,Recommended_Intervention
Diego,Low Average Score: 48,Comprehensive academic support to identify opportunities for improvement.
Daniel,Low Math Score: 48,Targeted Tutoring.
Javier,Low Average Score: 49,Comprehensive academic support to identify opportunities for improvement.
